# 03_3 · Chọn mô hình trên tập validation
**Phần 3/4.** Huấn luyện và so sánh ItemKNN, Latent (ALS hoặc SVD), Hybrid; tìm tham số tốt nhất theo NDCG@10 của nhóm khách có lịch sử; chọn chiến lược cho khách mới.

**Đầu ra:** `reports/recsys_validation.csv`, `reports/recsys_khach_moi.csv`, cấu hình tốt nhất lưu vào `data/rs_state.json`. Phần này chạy lâu nhất (vài phút đến vài chục phút tùy máy).

In [1]:
from recsys_lib import *
from IPython.display import display

inter, item_stats, catalog, n_inv_total = load_cache()
st = load_state(); blacklist = set(st["blacklist"])
print(f"{len(inter):,} dòng | loại {len(blacklist)} mã không phải hàng hóa")

Latent model: SVD (scikit-learn)
350,663 dòng | loại 34 mã không phải hàng hóa


In [2]:
val = Split(inter, catalog, blacklist, *VAL); print(val.summary())
print("Latent model:", "ALS (implicit)" if HAS_IMPLICIT else "SVD (scikit-learn)")

train < 01/11/2025: 209,897 dòng, 55,969 khách | mẫu được gợi ý: 1,429 | khách kỳ đánh giá: 9,426 (có lịch sử 3,066, mới 6,360) | trần Recall: 0.876
Latent model: SVD (scikit-learn)


## 6. Chọn mô hình và tham số trên tập validation (tháng 11)
Tiêu chí chọn: **NDCG@10 trên nhóm khách có lịch sử** (nhóm khách mới do mô hình riêng xử lý).
Lưới tham số nhỏ để chạy nhanh; mở rộng nếu còn thời gian.

In [3]:
import itertools, time
pop30 = Popularity(30, "buyers").fit(val); first60 = FirstTimer(60).fit(val)
results, models_val = [], {}

def run(name, model, cold=first60, **extra):
    t0 = time.time()
    res = evaluate(val, make_scorer(model, cold))
    df = summarize(res, name, extra); df["giay"] = time.time() - t0
    results.append(df); models_val[name] = (model, res); return df

run("Popularity30", pop30, cold=pop30)
run("SegmentPop", SegmentPopularity(30, 0.7).fit(val))
for k_, tau_ in itertools.product([20, 50], [60, 120]):
    run(f"ItemKNN k={k_} tau={tau_}", ItemKNN(k_, tau_).fit(val), k=k_, tau=tau_)
for f_, reg_, a_ in itertools.product([32, 64], [0.05, 0.5], [10, 40]):
    run(f"Latent f={f_} reg={reg_} a={a_}", Latent(f_, reg_, a_).fit(val), factors=f_, reg=reg_, alpha=a_)
tab = pd.concat(results)
rank = tab[tab.nhom == "co_lich_su"].sort_values(f"ndcg@{K}", ascending=False)
rank[["mo_hinh", f"recall@{K}", f"ndcg@{K}", f"hit_rate@{K}", "coverage", "da_dang_nganh"]].head(12)

,mo_hinh,recall@10,ndcg@10,hit_rate@10,coverage,da_dang_nganh
1,SegmentPop,0.0581,0.0375,0.1530,0.0217,4.5453
1,Popularity30,0.0406,0.0268,0.1099,0.0105,4.8477
1,Latent f=32 reg=0.5 a=40,0.0360,0.0251,0.1047,0.1428,4.0933
1,Latent f=32 reg=0.5 a=10,0.0360,0.0251,0.1047,0.1428,4.0933
1,Latent f=32 reg=0.05 a=40,0.0360,0.0251,0.1047,0.1428,4.0933
1,Latent f=32 reg=0.05 a=10,0.0360,0.0251,0.1047,0.1428,4.0933
1,Latent f=64 reg=0.05 a=10,0.0374,0.0249,0.1089,0.2043,3.9253
1,Latent f=64 reg=0.05 a=40,0.0374,0.0249,0.1089,0.2043,3.9253
1,Latent f=64 reg=0.5 a=10,0.0374,0.0249,0.1089,0.2043,3.9253
1,Latent f=64 reg=0.5 a=40,0.0374,0.0249,0.1089,0.2043,3.9253


In [4]:
# Trộn với độ phổ biến: thử các trọng số w cho hai mô hình tốt nhất (mỗi họ một mô hình)
best_knn = rank[rank.mo_hinh.str.startswith("ItemKNN")].iloc[0]; best_lat = rank[rank.mo_hinh.str.startswith("Latent")].iloc[0]
print("Tốt nhất ItemKNN:", best_knn.mo_hinh, "| tốt nhất Latent:", best_lat.mo_hinh)
for base_name in [best_knn.mo_hinh, best_lat.mo_hinh]:
    for w in [0.0, 0.1, 0.2, 0.4]:
        run(f"Hybrid[{base_name}] w={w}", Hybrid(models_val[base_name][0], pop30, w), w=w)
tab = pd.concat(results)
rank = tab[tab.nhom == "co_lich_su"].sort_values(f"ndcg@{K}", ascending=False)
rank[["mo_hinh", f"recall@{K}", f"ndcg@{K}", f"hit_rate@{K}", "coverage"]].head(10)

Tốt nhất ItemKNN: ItemKNN k=50 tau=120 | tốt nhất Latent: Latent f=32 reg=0.5 a=40


,mo_hinh,recall@10,ndcg@10,hit_rate@10,coverage
1,SegmentPop,0.0581,0.0375,0.1530,0.0217
1,Hybrid[Latent f=32 reg=0.5 a=40] w=0.4,0.0456,0.0318,0.1282,0.0763
1,Hybrid[Latent f=32 reg=0.5 a=40] w=0.2,0.0429,0.0289,0.1220,0.1057
1,Hybrid[ItemKNN k=50 tau=120] w=0.4,0.0402,0.0285,0.1213,0.9006
1,Hybrid[Latent f=32 reg=0.5 a=40] w=0.1,0.0402,0.0274,0.1155,0.1148
1,Popularity30,0.0406,0.0268,0.1099,0.0105
1,Hybrid[ItemKNN k=50 tau=120] w=0.2,0.0367,0.0263,0.1099,0.9342
1,Hybrid[ItemKNN k=50 tau=120] w=0.1,0.0353,0.0251,0.1053,0.9398
1,Hybrid[Latent f=32 reg=0.5 a=40] w=0.0,0.0360,0.0251,0.1047,0.1428
1,Latent f=32 reg=0.05 a=40,0.0360,0.0251,0.1047,0.1428


### Chiến lược cho khách mới

In [5]:
# Chiến lược cho khách MỚI (chưa có lịch sử): so sánh hai cách
cold_rows = []
for name, m in [("Popularity30 (toàn chuỗi)", pop30), ("FirstTimer60 (khách mới hay mua)", first60),
                ("Popularity14", Popularity(14).fit(val)), ("FirstTimer30", FirstTimer(30).fit(val)),
                ("FirstTimer90", FirstTimer(90).fit(val))]:
    r = evaluate(val, make_scorer(m)); df = summarize(r, name)
    cold_rows.append(df[df.nhom == "khach_moi"])
cold = pd.concat(cold_rows); cold[["mo_hinh", "so_khach", f"recall@{K}", f"ndcg@{K}", f"hit_rate@{K}"]]

,mo_hinh,so_khach,recall@10,ndcg@10,hit_rate@10
2,Popularity30 (toàn chuỗi),6360,0.0451,0.0274,0.0992
2,FirstTimer60 (khách mới hay mua),6360,0.0453,0.0289,0.0980
2,Popularity14,6360,0.0525,0.0300,0.1277
2,FirstTimer30,6360,0.0502,0.0296,0.1107
2,FirstTimer90,6360,0.0359,0.0219,0.0810


In [6]:
best = rank.iloc[0]
cold_best = cold.sort_values(f"ndcg@{K}", ascending=False).iloc[0].mo_hinh
tab.to_csv(REPORTS / "recsys_validation.csv", index=False); cold.to_csv(REPORTS / "recsys_khach_moi.csv", index=False)
save_state(best_config=best.mo_hinh, cold_choice=cold_best, best_val_ndcg=float(best[f"ndcg@{K}"]))
print("Cấu hình tốt nhất:", best.mo_hinh, "| chiến lược khách mới:", cold_best)

Cấu hình tốt nhất: SegmentPop | chiến lược khách mới: Popularity14


## Việc cần làm trước khi sang phần 4
Xem bảng xếp hạng: mô hình cá nhân hóa có hơn rõ rệt Popularity30 và SegmentPop không? Nếu không, kết luận trung thực của bạn là "độ phổ biến đã rất mạnh với dữ liệu này", và đó vẫn là một kết quả hợp lệ.